# 05 · Tensile-test analysis of ten S355 specimens: overview

**Goal.** Run the complete analysis of all ten tensile tests from the hosted research graphs, once from their PMDco 2 description and once from their PMDco 3 description, and show the results: properties per specimen, statistics per orientation, the comparison with the lab's values, and two overview figures.

**Audience.** All readers; the entry point for materials scientists and engineers (MSE experts).

**Prerequisites.** The terms of the tensile test (ISO 6892-1: E, ReH, ReL, Rp0.2, Fm, Rm, Ag, Agt, A, Z). No RDF or workflow knowledge is needed. The `pmd-demonstrator` environment from `environment.yml`, which installs the package `pmd_tensile`, and `ONTODOCKER_ADDRESS` (and `ONTODOCKER_TOKEN`, if the instance needs one) set in the environment before Jupyter starts. Neither is printed here.

**Reading paths.** MSE experts: **05** → 02 → 01. Ontologists: 01 → 04 → **05**. Workflow developers: 03 → 04 → 02 → **05**.

| Notebook | Details on |
| --- | --- |
| [01 · The tensile tests as data](01-tensile-data-from-graphs.ipynb) | the PMDco 2 and PMDco 3 graphs, ontology profiles, the neutral records, the Dataportal records |
| [02 · Analysis, node by node](02-tensile-analysis-nodes.ipynb) | what each analysis step computes and where it fails, on Zx3, Zy3 and Zd3 |
| [03 · Building the workflow](03-tensile-workflow-building.ipynb) | how the nodes form the per-specimen workflow and the batch, and how to change a method parameter |
| [04 · The workflow as a knowledge graph](04-tensile-workflow-knowledge-graph.ipynb) | what semantikon records about a run, validation, and a query across run and data graph |

All properties are **method-defined, ISO 6892-1-inspired** (method specification v2.6). Nothing here claims conformity with ISO 6892-1 or its Annex G.

*Execution class: network-read.* SPARQL queries to Ontodocker and the download of ten curve files from the public data host, once per ontology generation. Nothing is written to remote services or to disk.

## 1. Description and analysis

The research graphs say *what* was measured: which specimen was tested, its cross-section and gauge length, which CSV file holds its curve, which column is the force and in which unit. The analysis needs numbers and arrays in known units, not the concepts. The workflow therefore has a boundary:

```
graph (PMDco 2 or PMDco 3) --exploration--> specimen IDs
                           --acquisition--> neutral records --analysis--> properties, flags, uncertainty components
```

- **Exploration** and **acquisition** are the only steps that know an ontology. An *ontology profile*, selected by the workflow input `profile_id`, holds the ontology-specific SPARQL queries and unit conventions (notebook 01).
- **Analysis** steps receive plain numbers and arrays with units. They see no IRI and no SPARQL query (notebook 02).

Switching the ontology therefore means changing two inputs, `profile_id` and `source`. Section 3 checks that the results do not change.

## 2. Setup and the two runs

Select the `pmd-demonstrator` kernel. The next cell is tagged `parameters` for papermill. The sources are the three hosted datasets of each generation (orientation groups); both describe the same ten tests and link the same curve files.

In [ ]:
import os

import pandas as pd
from IPython.display import display

from pmd_tensile import analysis, demo, workflows

pd.set_option("display.max_colwidth", None)  # flags and lab notes are long
pd.set_option("display.precision", 4)

if not os.environ.get(demo.ONTODOCKER_ADDRESS_ENV, "").strip():
    raise RuntimeError("Set ONTODOCKER_ADDRESS (and ONTODOCKER_TOKEN if required) before starting Jupyter.")

In [ ]:
orientations = ("parallel", "perpendicular", "diagonal")
sources = {
    "pmdco2-tto": {"kind": "ontodocker", "datasets": [f"pmdco2_tto_example_{o}" for o in orientations]},
    "pmdco3-tto": {"kind": "ontodocker", "datasets": [f"pmdco3_tto_example_{o}" for o in orientations]},
}

`run_tensile_analysis` runs the two workflow stages: the exploration checks the source against the profile and lists the specimens; the batch runs one per-specimen workflow for each (acquisition, analysis, comparison with the lab values) and gathers the tables. Notebook 03 shows how the batch is built. The method parameters are the defaults of method specification v2.6. The timestamp taken just before the runs appears in the figures.

*Execution class: network-read (SPARQL queries; one curve download per specimen and run).*

In [ ]:
print("Method:", analysis.METHOD_LABEL)
print(analysis.E_LABEL)

retrieved = pd.Timestamp.now(tz="Europe/Berlin").floor("s")
tensile = {profile_id: demo.run_tensile_analysis(profile_id, source) for profile_id, source in sources.items()}
for profile_id, run in tensile.items():
    print(f"{profile_id}: batch run {run.batch_run.status}, {len(run.specimen_ids)} specimens ({', '.join(run.specimen_ids)})")
print("Data retrieved:", retrieved)

## 3. Same results from both ontology generations

The two runs differ only in `profile_id` and `source`. If the profiles map both descriptions to the same neutral records, and the curve bytes are the same, every result must be identical, down to the last digit, the flags and the lab values used in the comparison. The check compares the text representation of every cell of the three result tables (exact and NaN-aware) and the SHA-256 of each downloaded curve.

In [ ]:
pmdco2, pmdco3 = tensile["pmdco2-tto"], tensile["pmdco3-tto"]


def identical(left: pd.DataFrame, right: pd.DataFrame) -> bool:
    """Same columns, rows and cell values; repr makes the comparison exact and NaN-aware."""
    return (
        list(left.columns) == list(right.columns)
        and left.shape == right.shape
        and bool((left.reset_index(drop=True).map(repr) == right.reset_index(drop=True).map(repr)).all().all())
    )


display(pd.DataFrame([
    {"table": name, "rows": len(getattr(pmdco2, name)), "columns": getattr(pmdco2, name).shape[1],
     "identical": identical(getattr(pmdco2, name), getattr(pmdco3, name))}
    for name in ("results", "orientation_summary", "comparison")
]).set_index("table"))
same_curves = [sid for sid in pmdco2.specimen_ids if pmdco2.curves[sid].sha256 == pmdco3.curves[sid].sha256]
print("Same specimen IDs in the same order:", pmdco2.specimen_ids == pmdco3.specimen_ids)
print(f"Same curve bytes (SHA-256): {len(same_curves)} of {len(pmdco2.specimen_ids)} specimens")

The tables below and the figures are drawn from the PMDco 2 run; the PMDco 3 run gives the same numbers. This shows that the result does not depend on the ontology generation, not that it is right: the weak points discussed in section 8 appear in both.

## 4. Properties per specimen

Undefined values are NaN, and the flags name the rule that produced them; the flags and lab notes are shown in full. No specimen is excluded. Two entries need care (section 8): Ag and Agt are NaN on all ten specimens, because the strain record ends long before the force maximum; and Zy3 is listed as yielding continuously, with ReH and ReL NaN, although its curve has a clear upper yield peak.

*Execution class: read-only (local), as are all remaining cells.*

In [ ]:
result = pmdco2
results = result.results.set_index("specimen_id")
property_columns = ["orientation", "yield_type", "e_gpa", "reh_mpa", "rel_mpa", "rp02_mpa", "fm_n", "rm_mpa",
                    "ag_percent", "agt_percent", "a_percent", "z_percent"]
display(results[property_columns])
display(results[["notes", "flags"]])

### Uncertainty components

The components are reported separately and never combined. `s0_u_rel` is the relative standard uncertainty of S0 from the scatter of the cross-section measurements s1–s3; it applies equally to every stress-based value, including E, and is one to two orders of magnitude smaller than plausible force-class and E effects. The window ranges are the spread of E and Rp0.2 over a predeclared grid of fit windows: a sensitivity, not a confidence interval, and they never select a window. Panels (g) and (h) of the second figure show both. Not covered:

In [ ]:
display(results[["s0_u_rel", "e_gpa", "e_window_min_gpa", "e_window_max_gpa", "e_linearity_rel_diff",
                 "rp02_mpa", "rp02_window_min_mpa", "rp02_window_max_mpa"]])
print("S0 scatter:", analysis.S0_SCATTER_LABEL)
for item in analysis.OMITTED_UNCERTAINTY + analysis.RESIDUAL_RISKS:
    print("  not covered:", item)

## 5. Statistics per orientation

Descriptive statistics only: n is 2–4 per orientation, so there are no hypothesis tests and no claims about orientation effects. `n` counts the specimens with a defined value; `sd` uses ddof = 1. Zx1 was tested at a different speed (lab note).

The diagonal group has two specimens, and Zd3's E is the slope of a segment that is not straight (section 8), so the diagonal E mean (179.8 GPa) and sd (29.9 GPa) reflect that one record; they do not estimate the modulus in this orientation or its scatter. In the perpendicular group, ReH and ReL have n = 3 because Zy3 has no value, and the Rp0.2 statistics include Zy3's value, which is a plateau stress. Ag and Agt have n = 0 in every group.

In [ ]:
display(
    result.orientation_summary.pivot(index="quantity", columns=workflows.GROUP_COLUMN, values=["n", "mean", "sd"])
    .reindex(list(workflows.SUMMARY_QUANTITIES))
)

## 6. Comparison with the lab's values

Each computed value is compared with the value the lab reported in the graph, against tolerances fixed before any real data were analysed: Fm ±0.1 %, Rm ±(0.1 % + 0.5 MPa), ReH and Rp0.2 ±2 %, E ±10 %, A and Z rounding-aware. ReL is not compared, because the lab reports no values. The outcome is `within`, `outside`, or `not_comparable` when either value is missing.

This is a **consistency check with the lab's software, not a validation**: the lab values are not ground truth, and their evaluation windows are unknown. `within` only says that two evaluations of the same raw curve with the same S0 agree inside the tolerance; errors they share cannot show up. Fm and Rm depend on little more than the maximum of the force channel, and A and Z are arithmetic on the graph's metadata. The close agreement in E on nine specimens (the largest deviation is printed below, on values the lab rounds to 1 GPa) shows that both fits find the same slope, not that the slope is the material's modulus. `not_comparable` is not agreement.

In [ ]:
comparison = result.comparison
display(pd.crosstab(comparison["quantity"], comparison["outcome"]))
display(comparison[comparison["outcome"] != "within"][["specimen_id", "quantity", "computed", "reported", "outcome"]])
e_within = comparison[(comparison["quantity"] == "e_gpa") & (comparison["outcome"] == "within")]
print(f"E within tolerance on {len(e_within)} specimens; largest deviation from the lab value: "
      f"{100 * e_within['rel_deviation'].abs().max():.2f} %")

The raw differences, computed minus lab value in the unit of each quantity (percentage points for A and Z). Panel (e) of the second figure shows the same differences divided by their tolerances.

In [ ]:
UNITS = {"fm_n": "N", "rm_mpa": "MPa", "reh_mpa": "MPa", "rp02_mpa": "MPa", "e_gpa": "GPa", "a_percent": "%", "z_percent": "%"}
raw = comparison.assign(**{"computed − lab": comparison["computed"] - comparison["reported"], "lab": comparison["reported"]})
raw["quantity"] = raw["quantity"] + " / " + raw["quantity"].map(UNITS)
values = ["computed", "lab", "computed − lab"]
table = raw.pivot(index="quantity", columns="specimen_id", values=values).stack(level=0)
quantities = list(dict.fromkeys(raw["quantity"]))
display(table.reindex(pd.MultiIndex.from_product([quantities, values])).reindex(columns=list(result.specimen_ids)))
tolerances = comparison.drop_duplicates("quantity").set_index("quantity")["tolerance"]
print("Tolerances:", "; ".join(f"{quantity}: {tolerance}" for quantity, tolerance in tolerances.items()))

## 7. Overview figures

Both figures are drawn from the workflow's own results: the curves are the arrays the analysis used, and nothing is downloaded again. Colours mark specimens, line styles and markers mark orientations; the legend gives E, ReH and Rm per specimen ("–" where undefined) and the data-retrieval time. Curves are raw samples without smoothing.

**Curves.** (a) All ten curves up to the extensometer end, each shifted by its fitted strain origin e₀. (b) The yield region, with ReH, ReL and Rp0.2 marked. (c) The method on Zx3: the stress window of the elastic fit, the elastic line, the line shifted by 0.2 % plastic strain, and the extensometer end. (d) Stress against crosshead travel to fracture, with Fm marked.

In [ ]:
fig = demo.curves_overview_figure(result, retrieved=retrieved)

**Results.** (e) Computed minus lab value per specimen and quantity, in units of the tolerance, so the green band is `within`. (f) ReH against Rm. (g) E against Rp0.2, with the E range over the fit-window grid as bars. (h) Mean ± sd per orientation, with each specimen and its S0-based uncertainty (k = 1), not combined.

In [ ]:
fig = demo.results_overview_figure(result, retrieved=retrieved)

## 8. Limitations

- **ISO-inspired, not conformant.** All properties are method-defined, ISO 6892-1-inspired. Nothing here claims conformity with ISO 6892-1 or its Annex G, or certifies the material.
- **Extensometer, Ag and Agt.** On all ten records the strain column ends at about 1.9 % (Zx1), 3.9 % (Zx2–Zx4, Zy1–Zy4) or 4.2 % (Zd2, Zd3), at a pause in the test. The lab notes record removal of the extensometer after 3.9 % strain only for Zx2, Zx3 and Zx4; for the other seven, removal is our inference, not a statement of the graphs. The force maximum lies far beyond the extensometer end on every specimen, so **Ag and Agt are NaN on all ten**, with the flag `extensometer_invalid`, and are not estimated from the crosshead.
- **E is not a material modulus.** It is the regression slope of the elastic part (mE-type), not Annex-G-conformant. The extensometer class is unknown, which alone allows slope errors of about 10 %.
- **Zd3: E deviates because the elastic part is not straight.** The computed 158.7 GPa (lab 195 GPa, −18.6 %, `outside`) is reported as the method gives it, but it should not be read as a modulus. The curvature shows in R² = 0.9954, in `e_linearity_rel_diff` = −0.14 and in a window range of 152–180 GPa that does not contain the lab value. No flag marks this, because the method has no rule for it. The cause is unknown: settling or slip of the extensometer or grips, bending and local deformation all remain possible. How the lab obtained 195 GPa is also unknown.
- **Zy3: a real yield peak is missed.** The curve has a clear stress maximum of 407.6 MPa at 0.20 % strain, and the lab reports ReH 408 MPa, yet the method reports continuous yielding. A low-load event (the force falls by about 21 % within 0.07 s at about 34 MPa) lies inside the provisional fit window and raises the noise level to 4.9 MPa, so the required drop (24.4 MPa) exceeds the 17.5 MPa that occurs before the extensometer end. ReH and ReL are NaN, and the flag `reh:extensometer_invalid` does not name this cause. The reported Rp0.2 (395.8 MPa) is a plateau stress, although the flag `in_yield_plateau` is missing. The result is reported as the approved method gives it, because changing the method after seeing this comparison would make a new agreement with the lab uninformative. Zd3 passes the yield test narrowly (drop 6.5 MPa, required 5.5 MPa), so the weakness is not specific to Zy3.
- **Further weak points**, found on synthetic curves: a noisy plateau without an upper peak can be classified as discontinuous (a spurious ReH), and a record whose stress never returns to ReH gives a wrong ReL and no Fm, with the flag `fm:no_return_to_reh`; the line below names any specimen here with that flag.
- **Lab values are not ground truth**, and **the uncertainty is partial** (section 4).

Notebook 02 shows the Zy3 and Zd3 cases step by step.

In [ ]:
no_return = results.index[results["flags"].str.contains("fm:no_return_to_reh")]
print("Stress never returns to ReH (fm:no_return_to_reh):", ", ".join(no_return) or "no specimen")

## Summary

- The same workflow analysed all ten specimens from the PMDco 2 and the PMDco 3 graphs; with identical curve bytes, every value, flag and comparison is identical (section 3).
- E, ReH, Rp0.2, Fm and Rm are defined on all specimens except where a rule says otherwise (Zy3's ReH and ReL); Ag and Agt are undefined on all ten because the strain record ends early (sections 4 and 8).
- The comparison with the lab values is a consistency check; Zd3's E is `outside` and Zy3's ReH is `not_comparable` (section 6).

Where to continue: [01](01-tensile-data-from-graphs.ipynb) for the graphs and the profiles, [02](02-tensile-analysis-nodes.ipynb) for the analysis steps, [03](03-tensile-workflow-building.ipynb) for the workflow construction, [04](04-tensile-workflow-knowledge-graph.ipynb) for the run as a knowledge graph.